# Compensation by Group and Year — Template

For each **Job_Family × Compensation_Grade × Country (Location)** group, shows the
**average**, **minimum**, and **maximum** hire pay in each **year**.

The year is taken from `Hire_Date`. Edit the `CONFIG` cell to point at your data
and confirm the column-name mapping, then run all cells (Kernel ▸ Restart & Run All).

_Data source: ADM — Analytics Data Mart. Compensation field = `Hire_Pay` (USD)._

## 1. Configuration

In [ ]:
# ---- CONFIG: edit these to match your data ----

DATA_PATH = 'data.csv'        # path to ADM export (.csv, .xlsx, or .parquet)
SHEET_NAME = 0                # used only for .xlsx (sheet index or name)

# Map the logical fields used here -> the column names in YOUR data.
COLS = {
    'job_family':        'Job_Family',
    'compensation_grade':'Compensation_Grade',
    'country':           'Country',          # if you only have 'Location', map it here
    'compensation':      'Hire_Pay',         # the numeric pay column to analyze
    'hire_date':         'Hire_Date',        # year is extracted from this date column
}

# Group keys. Output is one row per group per year.
GROUP_KEYS = ['job_family', 'compensation_grade', 'country']

OUTPUT_XLSX = 'compensation_by_year.xlsx'
OUTPUT_CSV  = 'compensation_by_year.csv'

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 300)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

## 3. Load data

Supports CSV, Excel, and Parquet based on file extension.

In [ ]:
def load_data(path, sheet_name=0):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Could not find '{path}'. Update DATA_PATH in the CONFIG cell.")
    ext = p.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(p)
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(p, sheet_name=sheet_name)
    if ext == '.parquet':
        return pd.read_parquet(p)
    raise ValueError(f'Unsupported file type: {ext}')

df_raw = load_data(DATA_PATH, SHEET_NAME)
print(f'Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 4. Validate & prepare

Confirms mapped columns exist, coerces pay to numeric, and extracts the hire year.

In [ ]:
# Check every mapped column is present.
missing = [s for s in COLS.values() if s not in df_raw.columns]
if missing:
    raise KeyError(
        f'These mapped columns are not in the data: {missing}. '
        f'Available columns: {list(df_raw.columns)}')

# Standardize to the logical names used below.
df = df_raw.rename(columns={s: logical for logical, s in COLS.items()}).copy()

# Coerce pay to numeric and extract the hire year.
df['compensation'] = pd.to_numeric(df['compensation'], errors='coerce')
df['year'] = pd.to_datetime(df['hire_date'], errors='coerce').dt.year

print(f'Rows total:                        {len(df):,}')
print(f'Rows with non-numeric/missing pay: {df["compensation"].isna().sum():,}')
print(f'Rows with unparseable hire date:   {df["year"].isna().sum():,}')
print(f'Rows missing a group key:          {df[GROUP_KEYS].isna().any(axis=1).sum():,}')

# Keep rows usable for the analysis: valid pay, valid year, complete group keys.
df_valid = df.dropna(subset=['compensation', 'year'] + GROUP_KEYS).copy()
df_valid['year'] = df_valid['year'].astype(int)
print(f'Rows used in analysis:             {len(df_valid):,}')

## 5. Average / min / max pay by group and year

One row per **Job_Family × Compensation_Grade × Country × Year** with the average,
minimum, and maximum hire pay. `sample_size` shows how many hires each figure is based on.

In [ ]:
summary = (df_valid
           .groupby(GROUP_KEYS + ['year'], dropna=False)
           .agg(sample_size=('compensation', 'size'),
                avg_pay=('compensation', 'mean'),
                min_pay=('compensation', 'min'),
                max_pay=('compensation', 'max'))
           .reset_index()
           .sort_values(GROUP_KEYS + ['year'])
           .reset_index(drop=True))

print(f'{len(summary):,} group-year rows across {summary["year"].nunique()} years')
summary

## 6. Optional — single-group drill-down

Filter to one group to see its year-by-year history. Set any value to `None` to ignore that dimension.

In [ ]:
def drill(data, job_family=None, compensation_grade=None, country=None):
    mask = pd.Series(True, index=data.index)
    if job_family is not None:
        mask &= data['job_family'] == job_family
    if compensation_grade is not None:
        mask &= data['compensation_grade'] == compensation_grade
    if country is not None:
        mask &= data['country'] == country
    return data[mask]

# Example (edit to values present in your data):
# drill(summary, job_family='Engineering', country='United States')

## 7. Export

Writes the group-by-year table to Excel and CSV.

In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as xw:
    summary.to_excel(xw, sheet_name='pay_by_year', index=False)

summary.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_XLSX} and {OUTPUT_CSV}')